# ColorCorrect: chart to calibrated images

Audience:
- Researchers and developers who want the desktop processing pipeline without a GUI.

Prerequisites:
- Python 3.10+, Jupyter and Pillow for inline thumbnails (`pip install jupyterlab pillow`), a built/extracted ColorCorrect package, and the repository example images. Set COLORCORRECT_SAMPLES if the images live elsewhere. The SDK itself needs no pip dependencies.

Learning goals:
- Detect a chart; fit FFC → GC → WB → CC; inspect honest metrics; apply a saved model; export a LUT and stage preview. Outputs go to a fresh temporary directory, never over the originals.

Research: [original publication](https://acsess.onlinelibrary.wiley.com/doi/full/10.1002/ppj2.70067). See [API guide](../docs/api/README.md) for the complete protocol.


## Outline

1. Setup
2. Detect and fit with Blank.JPG used only as the flat-field reference
3. Apply, preview and export
4. Exercise: distinguish fitted-chart error from generalization


In [ ]:
import os, sys, tempfile
from pathlib import Path
root = Path(os.environ['COLORCORRECT_ROOT']).resolve() if os.environ.get('COLORCORRECT_ROOT') else next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'api/colorcorrect.py').is_file()), None)
assert root, 'Open this notebook from the ColorCorrect folder or notebooks/ subfolder'
sys.path.insert(0, str(root / 'api'))
from colorcorrect import Engine
samples = Path(os.environ.get('COLORCORRECT_SAMPLES', str(root / 'test_images'))).resolve()
chart, white = samples / 'ColorCard_Big.JPG', samples / 'Blank.JPG'
assert chart.is_file() and white.is_file(), 'Set COLORCORRECT_SAMPLES to the example corpus folder'
out = Path(tempfile.mkdtemp(prefix='colorcorrect-tutorial-'))
print('Output directory:', out)


## Detect, then fit

This uses CPU deliberately for a portable baseline. Blank.JPG describes lighting, not chart colors. FFC must be explicitly enabled again for Apply. GC is automatic, without a degree control. The reference chart uses sRGB/D65 here. Detection and a full-resolution fit can take tens of seconds.


In [ ]:
from time import perf_counter
with Engine(root=root, policy='cpu') as engine:
    t = perf_counter()
    detection = engine.detect(chart, handshake_out=out / 'chart.json')
    fit = engine.fit(chart, handshake=detection['handshake'], output=out / 'chart_corrected.png',
        report=out / 'report.json', model_out=out / 'model.json', model='rpcc2',
        white=white, do_ffc=True, do_gc=True, do_wb=True, do_cc=True,
        ffc={'method': 'pls', 'degree': 5}, cross_validate=True, phase2_teacher=False)
    print('Chart:', detection['chart_type'], 'Elapsed seconds:', round(perf_counter() - t, 2))
report = fit['report_json']
print('Fitted-chart DeltaE00:', report.get('delta_e_2000'))
print('Held-out validation:', report.get('validation_delta_e_2000'))
assert report.get('do_ffc') is True, 'Inspect FFC warnings: white correction was skipped'


## Apply, preview and export

Use only images captured with compatible lighting, exposure and geometry. A LUT is a global color transform and cannot include spatial FFC. Apply reports intentionally do not invent DeltaE for images without measured chart patches.


In [ ]:
with Engine(root=root, policy='cpu') as engine:
    applied = engine.apply(out / 'model.json', [samples / 'Sample_3572.JPG', samples / 'Sample_3573.JPG'],
        out / 'applied', output_format='png', do_ffc=True, white=white)
    preview = engine.preview(out / 'model.json', chart, 'gc', out / 'preview', do_ffc=True, white=white)
    engine.preview(out / 'model.json', chart, 'ffc', out / 'ffc_preview', do_ffc=True, white=white)
    cube = engine.export_cube(out / 'model.json', out / 'grade.cube', size=33)
assert all(Path(row['output']).is_file() for row in applied['results'])
assert Path(cube['path']).is_file()
print('Applied images:', len(applied['results']), '| LUT:', cube['note'])


## Inspect the actual results
These are thumbnails of this run's real images, not mockups. The first comparison isolates spatial flat-field correction; the second shows the full pipeline. A visually pleasing result is not a substitute for the measured chart and held-out errors. Thumbnail compression is display-only and never changes your exported images.


In [ ]:
from io import BytesIO
from PIL import Image as PillowImage
from IPython.display import Image, Markdown, display
def show_thumbnail(path, caption):
    with PillowImage.open(path) as original:
        thumb = original.convert('RGB')
        thumb.thumbnail((640, 420))
        buffer = BytesIO()
        thumb.save(buffer, format='JPEG', quality=88)
    display(Markdown('**' + caption + '**'), Image(data=buffer.getvalue()))
show_thumbnail(chart, 'Original chart capture')
show_thumbnail(out / 'ffc_preview/ffc.png', 'After FFC only — Blank.JPG supplies the lighting reference')
show_thumbnail(out / 'chart_corrected.png', 'After FFC → GC → WB → CC')


## Exercise and pitfalls
Which metric should guide model choice: training or held-out error? The answer is held-out error, subject to the small chart's limits. Neither validates arbitrary scene pixels. Try the second notebook to compare fit methods. With sRGB/D65 input, embedded ICC profiles normalize to working sRGB; explicit other inputs bypass this conversion and must match their stored pixels. Outputs do not receive a matching ICC profile automatically.


In [ ]:
# Answer scaffold: inspect validation separately rather than assuming training improvement generalizes.
print('Training mean after:', report['delta_e_2000'].get('mean_after'))
print('Validation available:', 'validation_delta_e_2000' in report)
print('Warnings:', report.get('fit_warnings', []))
